# FaceXplain — face-masking test (≈5 minutes, no retraining)

Loads the 3 VGG16 models already trained by `FaceXplain_clean_rerun` and re-evaluates the **same 300 test images** under three conditions:

1. **Original** image
2. **Face only** — everything outside a central face ellipse is replaced with grey
3. **Background only** — the face ellipse is replaced with grey, only borders/background remain

If accuracy stays high on *Face only* and drops to ~50% on *Background only*, the model uses facial information. If *Background only* still scores well above 50%, the model relies on non-facial shortcuts.

Run with **Runtime → Run all** (T4 GPU).

In [ ]:
OUT_DIR = "/content/drive/MyDrive/FaceXplain_clean_rerun"   # same folder as the main run
SEEDS   = [42, 123, 2024]
IMG_SIZE = 224
BATCH_SIZE = 32
# Face ellipse, as a fraction of image width/height (images in this dataset are tight face crops)
ELLIPSE_RX, ELLIPSE_RY = 0.40, 0.48
GREY = (124, 116, 104)   # ImageNet mean colour, i.e. "neutral" for VGG16

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, roc_auc_score

split = pd.read_csv(os.path.join(OUT_DIR, "final_split.csv"))
test = split[split.split == "test"].reset_index(drop=True)
te_p, te_y = test.path.values, test.label.values
print("Test images:", len(te_p), "| ASD:", int(te_y.sum()), "| Non-ASD:", int((te_y == 0).sum()))

# same architecture as the main notebook
def build_vgg16():
    base = keras.applications.VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    aug = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.05), layers.RandomZoom(0.1),
                            layers.RandomContrast(0.1), layers.RandomBrightness(0.1)], name="augmentation")
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = keras.applications.vgg16.preprocess_input(aug(inp))
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Dense(128, activation="relu", name="head_dense")(x)
    x = layers.Dropout(0.5, name="head_dropout")(x)
    out = layers.Dense(1, activation="sigmoid", name="head_out")(x)
    model = keras.Model(inp, out)
    base.trainable = True
    for l in base.layers:
        l.trainable = l.name.startswith("block5")
    return model

In [ ]:
# masks
yy, xx = np.mgrid[0:IMG_SIZE, 0:IMG_SIZE]
c = (IMG_SIZE - 1) / 2
face = (((xx - c) / (ELLIPSE_RX * IMG_SIZE)) ** 2 + ((yy - c) / (ELLIPSE_RY * IMG_SIZE)) ** 2) <= 1.0
grey = np.array(GREY, dtype=np.float32)

def load_all(paths):
    imgs = []
    for p in paths:
        img = tf.io.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
        imgs.append(tf.image.resize(img, [IMG_SIZE, IMG_SIZE]).numpy())
    return np.stack(imgs).astype(np.float32)

X = load_all(te_p)
conditions = {
    "Original":        X,
    "Face only":       np.where(face[None, :, :, None], X, grey),
    "Background only": np.where(face[None, :, :, None], grey, X),
}
print(f"Face ellipse covers {face.mean()*100:.0f}% of the image")

fig, ax = plt.subplots(3, 4, figsize=(11, 8.5))
for r, (name, arr) in enumerate(conditions.items()):
    for k, i in enumerate([0, 1, 150, 151]):
        ax[r, k].imshow(arr[i].astype(np.uint8)); ax[r, k].axis("off")
    ax[r, 0].set_title(name, loc="left", fontsize=11)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "masking_examples.png"), dpi=150); plt.show()

In [ ]:
rows = []
for s in SEEDS:
    model = build_vgg16()
    model.load_weights(os.path.join(OUT_DIR, "weights", f"VGG16_seed{s}.weights.h5"))
    thr = json.load(open(os.path.join(OUT_DIR, "runs", f"VGG16_seed{s}.json")))["row"]["threshold"]
    for name, arr in conditions.items():
        p = model.predict(arr, batch_size=BATCH_SIZE, verbose=0).ravel()
        rows.append({"seed": s, "condition": name, "threshold": thr,
                     "accuracy": accuracy_score(te_y, (p >= thr).astype(int)),
                     "auc": roc_auc_score(te_y, p),
                     "mean_P_ASD": float(p.mean())})
    keras.backend.clear_session()

res = pd.DataFrame(rows)
summary = res.groupby("condition", sort=False)[["accuracy", "auc"]].agg(["mean", "std"])
print("Per seed:"); print(res.round(4).to_string(index=False))
print("\nMean ± std over seeds:")
for cond, r in summary.iterrows():
    print(f"  {cond:<16} accuracy {r[('accuracy','mean')]*100:5.1f} ± {r[('accuracy','std')]*100:.1f}%   "
          f"AUC {r[('auc','mean')]:.3f} ± {r[('auc','std')]:.3f}")
res.to_csv(os.path.join(OUT_DIR, "masking_test_results.csv"), index=False)
print("\nSaved:", os.path.join(OUT_DIR, "masking_test_results.csv"))